# Recall@10 Evaluation — replay against the 10 official trace files

Ground truth = the final table (end_of_conversation: true) in each trace.
We replay the trace's own scripted user lines against our live /chat,
using OUR agent's real replies as conversation history (not the trace's
canned agent replies), then compare our final recommendations to the
trace's final table.

In [1]:
import re
import glob
import time
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
TRACE_DIR = Path("../../traces")  # C1.md ... C10.md at repo root
REQUEST_TIMEOUT = 60

## Parse a trace .md into (ordered user messages, expected shortlist URLs)

In [2]:
def parse_trace(text: str):
    # Split into turn blocks
    turn_blocks = re.split(r"### Turn \d+", text)[1:]  # drop preamble before first turn

    user_messages = []
    final_urls = []
    final_eoc = False

    for block in turn_blocks:
        # User line: "> quoted text" right after **User**
        user_match = re.search(r"\*\*User\*\*\s*\n+>\s*(.+?)(?:\n\n|\Z)", block, re.DOTALL)
        if user_match:
            user_text = user_match.group(1)
            # Strip leading "> " continuation markers on multi-line quotes
            user_text = re.sub(r"^\>\s*", "", user_text, flags=re.MULTILINE).strip()
            user_messages.append(user_text)

        # end_of_conversation flag for this turn
        eoc_match = re.search(r"end_of_conversation.*?\*\*(true|false)\*\*", block, re.IGNORECASE)
        is_final = eoc_match and eoc_match.group(1).lower() == "true"

        # Table rows: markdown table lines, extract URL column (last | ... |)
        table_rows = re.findall(r"\|\s*\d+\s*\|.+?<(https://[^\s|>]+)>\s*\|", block)

        if is_final and table_rows:
            final_urls = table_rows
            final_eoc = True

    return user_messages, final_urls, final_eoc


def normalize_url(url: str) -> str:
    return url.rstrip("/").replace("/solutions", "").lower()

## Load all traces

In [3]:
trace_files = sorted(glob.glob(str(TRACE_DIR / "C*.md")))
print(f"Found {len(trace_files)} trace files in {TRACE_DIR.resolve()}")

traces = []
for path in trace_files:
    with open(path, encoding="utf-8") as f:
        text = f.read()
    user_msgs, expected_urls, has_final = parse_trace(text)
    if not has_final or not expected_urls:
        print(f"⚠️  {path}: no final confirmed shortlist found — skipping")
        continue
    traces.append({
        "file": path,
        "user_messages": user_msgs,
        "expected_urls": [normalize_url(u) for u in expected_urls],
    })
    print(f"{path}: {len(user_msgs)} user turns, {len(expected_urls)} expected items")

Found 10 trace files in C:\Users\DELL\Documents\GitHub\Recomendation_System\traces
..\..\traces\C1.md: 4 user turns, 3 expected items
..\..\traces\C10.md: 3 user turns, 2 expected items
..\..\traces\C2.md: 3 user turns, 5 expected items
..\..\traces\C3.md: 5 user turns, 4 expected items
..\..\traces\C4.md: 3 user turns, 5 expected items
..\..\traces\C5.md: 3 user turns, 5 expected items
..\..\traces\C6.md: 3 user turns, 2 expected items
..\..\traces\C7.md: 4 user turns, 5 expected items
..\..\traces\C8.md: 3 user turns, 5 expected items
..\..\traces\C9.md: 7 user turns, 7 expected items


## Replay a trace against the live /chat endpoint

Uses the trace's own scripted user lines, but OUR agent's real replies
as assistant history (the trace's original agent text is only used as
ground truth for the final list, never fed back in).

In [4]:
def replay_trace(user_messages, base_url=BASE_URL, max_extra_rounds=2):
    history = []
    last_recs = None
    last_eoc = False

    all_lines = list(user_messages)
    round_count = 0

    while all_lines or (not last_eoc and round_count < len(user_messages) + max_extra_rounds):
        if all_lines:
            next_user_line = all_lines.pop(0)
        else:
            # Our agent asked something the trace didn't anticipate — end gracefully
            break

        history.append({"role": "user", "content": next_user_line})

        for attempt in range(3):
            resp = requests.post(f"{base_url}/chat", json={"messages": history}, timeout=REQUEST_TIMEOUT)
            if resp.status_code in (429, 500, 503):
                wait = 20 * (attempt + 1)
                print(f"    retrying after {wait}s (status {resp.status_code})")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            break
        result = resp.json()

        history.append({"role": "assistant", "content": result.get("reply", "")})

        if result.get("recommendations"):
            last_recs = result["recommendations"]
        last_eoc = result.get("end_of_conversation", False)

        round_count += 1
        time.sleep(15)  # Gemini free-tier RPM pacing

        if last_eoc:
            break

    return last_recs or [], last_eoc, history

## Recall computation

In [5]:
def recall_at_k(predicted_urls, relevant_urls, k=10):
    predicted = set(predicted_urls[:k])
    relevant = set(relevant_urls)
    if not relevant:
        return 0.0
    return len(predicted & relevant) / len(relevant)

## Run full replay evaluation across all traces

**NOTE:** make sure `uvicorn src.main:app --reload` is running before executing the next cell.

In [6]:
results = []
for t in traces:
    print(f"\n=== Replaying {t['file']} ===")
    recs, reached_end, history = replay_trace(t["user_messages"])
    predicted_urls = [normalize_url(r.get("url", "")) for r in recs]

    r = recall_at_k(predicted_urls, t["expected_urls"], k=10)
    results.append({
        "trace": t["file"],
        "recall": r,
        "reached_end_of_conversation": reached_end,
        "num_predicted": len(predicted_urls),
        "num_expected": len(t["expected_urls"]),
        "predicted_urls": predicted_urls,
        "expected_urls": t["expected_urls"],
    })
    print(f"  Recall@10: {r:.2f}  (reached end: {reached_end})")


=== Replaying ..\..\traces\C1.md ===


  Recall@10: 0.67  (reached end: True)

=== Replaying ..\..\traces\C10.md ===
  Recall@10: 0.00  (reached end: False)

=== Replaying ..\..\traces\C2.md ===
  Recall@10: 0.00  (reached end: False)

=== Replaying ..\..\traces\C3.md ===
  Recall@10: 0.00  (reached end: False)

=== Replaying ..\..\traces\C4.md ===
  Recall@10: 0.00  (reached end: False)

=== Replaying ..\..\traces\C5.md ===
  Recall@10: 0.00  (reached end: False)

=== Replaying ..\..\traces\C6.md ===


KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(results)
print(f"\n{'='*60}")
print(f"MEAN RECALL@10 (trace replay): {results_df['recall'].mean():.4f}")
print(f"{'='*60}")
results_df[["trace", "recall", "reached_end_of_conversation", "num_predicted", "num_expected"]]

## Diagnostic: traces that didn't fully converge

In [ ]:
weak = results_df[results_df["recall"] < 1.0]
for _, row in weak.iterrows():
    print(f"\n{row['trace']}  (recall={row['recall']:.2f})")
    print(f"  Predicted: {row['predicted_urls']}")
    print(f"  Expected:  {row['expected_urls']}")